# 00 · Data Pipeline & Sequence Construction
**Owner:** Project Manager (Hessam)  
**Environment:** Local Anaconda (CPU — no GPU needed)  
**Output:** `data/processed/` — shared `.npy` tensors consumed by all model notebooks

---
## Purpose
This notebook is the single source of truth for all data preparation.  
Every model pair (Pair 1 / 2 / 3) loads the files produced here — **do not re-run preprocessing inside model notebooks.**

In [1]:
## ── 0. Environment check ──────────────────────────────────────────────────
import os, sys
print("Python:", sys.version)
print("Working dir:", os.getcwd())

# Detect execution environment
IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/FraudGuard'
else:
    # Resolve relative to this notebook's location
    BASE = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

DATA_RAW  = os.path.join(BASE, 'data', 'raw')
DATA_PROC = os.path.join(BASE, 'data', 'processed')
os.makedirs(DATA_PROC, exist_ok=True)
print(f"BASE       : {BASE}")
print(f"Raw data   : {DATA_RAW}")
print(f"Processed  : {DATA_PROC}")

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Working dir: C:\Users\hessa\OneDrive\Dropbox Backup\WorkSpace\GBC_AIML_Dev_2026\Applied Math for DP I\FraudGuard\notebooks
BASE       : C:\Users\hessa\OneDrive\Dropbox Backup\WorkSpace\GBC_AIML_Dev_2026\Applied Math for DP I\FraudGuard
Raw data   : C:\Users\hessa\OneDrive\Dropbox Backup\WorkSpace\GBC_AIML_Dev_2026\Applied Math for DP I\FraudGuard\data\raw
Processed  : C:\Users\hessa\OneDrive\Dropbox Backup\WorkSpace\GBC_AIML_Dev_2026\Applied Math for DP I\FraudGuard\data\processed


## 1 · Dataset Download Instructions
> **Action required before running cells below.**

1. Visit: https://www.kaggle.com/competitions/ieee-fraud-detection/data  
2. Download `train_transaction.csv` and `train_identity.csv`  
3. Place both files in:  `FraudGuard/data/raw/`  
4. Verify with the cell below before continuing.

In [2]:
## ── 1.1 Verify raw files exist ────────────────────────────────────────────
import os
needed = ['train_transaction.csv', 'train_identity.csv']
for f in needed:
    path = os.path.join(DATA_RAW, f)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else "MISSING"
    print(f"  {'✓' if exists else '✗'}  {f}  ({size})")
assert all(os.path.exists(os.path.join(DATA_RAW,f)) for f in needed), \
    "Missing files. Download from Kaggle and place in data/raw/ before continuing."

  ✓  train_transaction.csv  (683.4 MB)
  ✓  train_identity.csv  (26.5 MB)


## 2 · Load Raw Data

In [3]:
## ── 2.1 Load & join ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np

tx = pd.read_csv(os.path.join(DATA_RAW, 'train_transaction.csv'))
id_ = pd.read_csv(os.path.join(DATA_RAW, 'train_identity.csv'))

print(f"Transactions : {tx.shape}")
print(f"Identity     : {id_.shape}")

# Left-join — not all transactions have identity rows
df = tx.merge(id_, on='TransactionID', how='left')
print(f"Merged       : {df.shape}")
print(f"Fraud rate   : {df['isFraud'].mean()*100:.2f}%")
df.head(3)

Transactions : (590540, 394)
Identity     : (144233, 41)
Merged       : (590540, 434)
Fraud rate   : 3.50%


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3 · Exploratory Data Analysis

In [ ]:
## ── 3.1 Class distribution ────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class balance
counts = df['isFraud'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['#378ADD','#E24B4A'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontsize=9)

# Transaction amount by class
df[df['isFraud']==0]['TransactionAmt'].clip(0,1000).hist(ax=axes[1], bins=50,
    alpha=0.6, color='#378ADD', label='Legit')
df[df['isFraud']==1]['TransactionAmt'].clip(0,1000).hist(ax=axes[1], bins=50,
    alpha=0.7, color='#E24B4A', label='Fraud')
axes[1].set_title('Transaction Amount')
axes[1].legend()

# D1 (days since last transaction)
df[df['isFraud']==0]['D1'].clip(0,200).hist(ax=axes[2], bins=50,
    alpha=0.6, color='#378ADD', label='Legit')
df[df['isFraud']==1]['D1'].clip(0,200).hist(ax=axes[2], bins=50,
    alpha=0.7, color='#E24B4A', label='Fraud')
axes[2].set_title('D1 — Days Since Last Transaction')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(DATA_PROC, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()
print("Saved: data/processed/eda_overview.png")

## 4 · Cardholder UID Construction

In [4]:
## ── 4.1 Build UID — card1 + addr1 ────────────────────────────────────────
# Fill NaN before concatenation so UIDs don't collapse to NaN
df['card1']  = df['card1'].fillna(-999).astype(int).astype(str)
df['addr1']  = df['addr1'].fillna(-999).astype(int).astype(str)
df['uid']    = df['card1'] + '_' + df['addr1']

print(f"Unique UIDs  : {df['uid'].nunique():,}")
print(f"Mean tx/UID  : {df.groupby('uid').size().mean():.1f}")
print(f"Max  tx/UID  : {df.groupby('uid').size().max()}")
print()
# Validate: UID count vs card1-only count
print(f"Unique card1 only : {df['card1'].nunique():,}")
print(f"Unique card1+addr1: {df['uid'].nunique():,}  (← preferred)")

Unique UIDs  : 39,974
Mean tx/UID  : 14.8
Max  tx/UID  : 9928

Unique card1 only : 13,553
Unique card1+addr1: 39,974  (← preferred)


## 5 · Sort & Compute Delta-t

In [5]:
## ── 5.1 Sort by TransactionDT within each UID ────────────────────────────
df = df.sort_values(['uid', 'TransactionDT']).reset_index(drop=True)

# Delta-t in seconds (diff per cardholder group)
df['delta_t'] = df.groupby('uid')['TransactionDT'].diff().fillna(0)
df['delta_t']  = df['delta_t'].clip(lower=0)   # safety clip

print("delta_t sample (first 10 rows of one UID):")
uid_example = df['uid'].iloc[0]
print(df[df['uid']==uid_example][['TransactionDT','delta_t','TransactionAmt','isFraud']].head(10))

delta_t sample (first 10 rows of one UID):
   TransactionDT  delta_t  TransactionAmt  isFraud
0        4050851      0.0            29.0        0


## 6 · Feature Selection & Engineering

In [10]:
## ── 6.1 Define 15 features per timestep ──────────────────────────────────
# Core 15 features used by all model notebooks
FEATURES = [
    'TransactionAmt',   # raw amount
    'delta_t',          # seconds since previous tx in this sequence
    'D1',               # days since last tx (Vesta pre-computed)
    'C1', 'C2', 'C3', 'C5',          # count features
    'M1', 'M3', 'M6',                 # match flags (binary)
    'V12', 'V20', 'V37',              # top Vesta features
    'addr1_norm',                     # normalised billing zip
    'card_type_enc',                  # card4 label encoded
]
# Convert M-flag columns from 'T'/'F' strings → binary 1/0
M_COLS = ['M1', 'M3', 'M6']
for col in M_COLS:
    if col in df.columns:
        df[col] = df[col].map({'T': 1, 'F': 0}).fillna(0).astype(np.int8)

print("M-flag columns converted:")
print(df[M_COLS].value_counts())
# Derived features
df['addr1_norm'] = pd.to_numeric(df['addr1'], errors='coerce').fillna(0)
df['card_type_enc'] = df['card4'].map(
    {'visa':0,'mastercard':1,'american express':2,'discover':3}
).fillna(-1)

# Fill remaining NaNs
for col in FEATURES:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median() if df[col].dtype != object else 0)

print("Feature null counts after fill:")
print(df[FEATURES].isnull().sum())

M-flag columns converted:
M1  M3  M6
0   0   0     218933
1   1   0     139029
        1     112702
0   0   1      52192
1   0   0      39254
        1      28430
Name: count, dtype: int64
Feature null counts after fill:
TransactionAmt    0
delta_t           0
D1                0
C1                0
C2                0
C3                0
C5                0
M1                0
M3                0
M6                0
V12               0
V20               0
V37               0
addr1_norm        0
card_type_enc     0
dtype: int64


## 7 · Sliding Window Sequence Construction

In [11]:
## ── 7.1 Build sequences ───────────────────────────────────────────────────
SEQ_LEN = 5   # window size — agreed across all model pairs

def build_sequences(df, feature_cols, seq_len=5, min_seq=3):
    """
    For each cardholder UID, slide a window of `seq_len` transactions.
    Label = isFraud of the LAST transaction in the window.
    Cardholders with fewer than `min_seq` transactions are dropped.
    """
    X_list, y_list = [], []
    for uid, grp in df.groupby('uid', sort=False):
        grp = grp.reset_index(drop=True)
        if len(grp) < min_seq:
            continue
        # Pad short histories with zeros up to seq_len
        feats = grp[feature_cols].values.astype(np.float32)
        labels = grp['isFraud'].values
        for end in range(seq_len, len(grp) + 1):
            start = end - seq_len
            X_list.append(feats[start:end])
            y_list.append(labels[end - 1])
        # Handle sequences shorter than seq_len (near start of history)
        if len(grp) < seq_len:
            pad = np.zeros((seq_len - len(grp), len(feature_cols)), dtype=np.float32)
            seq = np.vstack([pad, feats])
            X_list.append(seq)
            y_list.append(labels[-1])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)

print(f"Building sequences (SEQ_LEN={SEQ_LEN})... this may take 2-3 min")
X, y = build_sequences(df, FEATURES, seq_len=SEQ_LEN)
print(f"X shape : {X.shape}   (samples, seq_len, features)")
print(f"y shape : {y.shape}")
print(f"Fraud rate in sequences: {y.mean()*100:.2f}%")

Building sequences (SEQ_LEN=5)... this may take 2-3 min
X shape : (498109, 5, 15)   (samples, seq_len, features)
y shape : (498109,)
Fraud rate in sequences: 3.64%


## 8 · Train / Val / Test Split

In [12]:
## ── 8.1 Stratified split ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

# 70 / 15 / 15  stratified by fraud label
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30,
                                              random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50,
                                                  random_state=42, stratify=y_tmp)
print(f"Train : {X_tr.shape}  fraud={y_tr.mean()*100:.1f}%")
print(f"Val   : {X_val.shape}  fraud={y_val.mean()*100:.1f}%")
print(f"Test  : {X_test.shape}  fraud={y_test.mean()*100:.1f}%")

Train : (348676, 5, 15)  fraud=3.6%
Val   : (74716, 5, 15)  fraud=3.6%
Test  : (74717, 5, 15)  fraud=3.6%


## 9 · Normalisation

In [13]:
## ── 9.1 MinMax scale per feature (fit on train only) ─────────────────────
from sklearn.preprocessing import MinMaxScaler
import pickle

n_samples, n_steps, n_feats = X_tr.shape

# Reshape to 2D, fit, reshape back
scaler = MinMaxScaler()
X_tr_2d  = scaler.fit_transform(X_tr.reshape(-1, n_feats))
X_val_2d = scaler.transform(X_val.reshape(-1, n_feats))
X_test_2d= scaler.transform(X_test.reshape(-1, n_feats))

X_tr   = X_tr_2d.reshape(n_samples, n_steps, n_feats)
X_val  = X_val_2d.reshape(X_val.shape[0], n_steps, n_feats)
X_test = X_test_2d.reshape(X_test.shape[0], n_steps, n_feats)

# Save scaler — MUST be used for inference in Streamlit
scaler_path = os.path.join(DATA_PROC, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved:", scaler_path)

Scaler saved: C:\Users\hessa\OneDrive\Dropbox Backup\WorkSpace\GBC_AIML_Dev_2026\Applied Math for DP I\FraudGuard\data\processed\scaler.pkl


## 10 · Class Weights & Save All Outputs

In [15]:
## ── 10.1 Compute class weights ────────────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_tr)
class_weights = {0: float(cw[0]), 1: float(cw[1])}
print(f"Class weights: {class_weights}")

## ── 10.2 Save tensors ─────────────────────────────────────────────────────
import json

np.save(os.path.join(DATA_PROC, 'X_train.npy'), X_tr)
np.save(os.path.join(DATA_PROC, 'X_val.npy'),   X_val)
np.save(os.path.join(DATA_PROC, 'X_test.npy'),  X_test)
np.save(os.path.join(DATA_PROC, 'y_train.npy'), y_tr)
np.save(os.path.join(DATA_PROC, 'y_val.npy'),   y_val)
np.save(os.path.join(DATA_PROC, 'y_test.npy'),  y_test)

with open(os.path.join(DATA_PROC, 'class_weights.json'), 'w') as f:
    json.dump(class_weights, f)

meta = {'seq_len': SEQ_LEN, 'features': FEATURES, 'n_features': len(FEATURES),
        'train_samples': int(X_tr.shape[0]), 'val_samples': int(X_val.shape[0]),
        'test_samples':  int(X_test.shape[0]), 'fraud_rate_train': float(y_tr.mean())}
with open(os.path.join(DATA_PROC, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print("\n=== All outputs saved to data/processed/ ===")
for fname in os.listdir(DATA_PROC):
    fpath = os.path.join(DATA_PROC, fname)
    size = os.path.getsize(fpath) / 1e6
    print(f"  {fname:30s}  {size:.1f} MB")


Class weights: {0: 0.5189063374328813, 1: 13.723079345088161}

=== All outputs saved to data/processed/ ===
  class_weights.json              0.0 MB
  meta.json                       0.0 MB
  scaler.pkl                      0.0 MB
  X_test.npy                      22.4 MB
  X_train.npy                     104.6 MB
  X_val.npy                       22.4 MB
  y_test.npy                      0.3 MB
  y_train.npy                     1.4 MB
  y_val.npy                       0.3 MB
